# Chapter 3 - Lab 9: <font color='blue'>PydanticAI Agent</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using PydanticAI** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

PydanticAI takes a **type-safe, Pythonic** stance: agents are declared as module-level objects, tools are registered with a `@agent.tool` decorator, and the LLM provider is specified as a single string. Pydantic-style validation runs at every boundary.

In this lab you will see how concise a tool-using agent can be when the framework treats Pydantic types as first-class citizens.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **PydanticAI** (`pydantic-ai`) — `Agent`, `RunContext`, `@agent.tool`.
* **OpenAI** `gpt-4o-mini` (via the `'openai:gpt-4o-mini'` provider string).
* **Pydantic** — used both for tool I/O and for output validation.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [ ]:
%pip install -q pydantic-ai pydantic python-dotenv

## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`OPENAI_API_KEY`** — your OpenAI key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

## 4. Declare the agent

PydanticAI agents are declared at module scope. The first argument names the model provider; the `system_prompt` is just a string. Notice how short this is compared to the AutoGen or ADK setup.

In [ ]:
from pydantic_ai import Agent, RunContext

agent = Agent('openai:gpt-4o-mini', system_prompt=system_message)

## 5. Register tools with `@agent.tool`

Tools are registered against a specific agent instance. The first parameter is always a `RunContext` (which carries deps, retries, model info), even if your tool does not need it. Return types may be Pydantic models or plain JSON-serialisable values.

In [ ]:
@agent.tool
def get_stock_data_tool(ctx: RunContext, ticker: str) -> dict:
    """Get stock data for a ticker."""
    result = get_stock_data(ticker)
    return {'ticker': ticker.upper(), 'price': result.price, 'eps': result.eps}


@agent.tool
def compute_pe_ratio_tool(ctx: RunContext, price: float, eps: float) -> float:
    """Compute the P/E ratio."""
    return compute_pe(price, eps)

## 6. Run the agent

`agent.run_sync` blocks until completion and returns a result whose `.output` holds the model's final answer. For async use `agent.run`.

In [ ]:
result = agent.run_sync(input_message)
print(result.output)

## 7. Results

You should see the agent call `get_stock_data_tool` (once per ticker), then `compute_pe_ratio_tool` (once per ticker), and finally produce a short memo comparing the two.

**What to notice about PydanticAI specifically:**

* The **most Pythonic** of the frameworks in this chapter — the agent is a regular module-level object you can configure, share, and test like any other.
* Pydantic types appear at every boundary (system prompt schema, tool I/O, validated outputs), which is excellent for production reliability.
* The `'provider:model'` string convention (also used by LangChain) means provider swaps are one-line changes.
* Trade-off: smaller community than LangChain or LlamaIndex; fewer pre-built integrations. Pick PydanticAI when type safety and explicitness matter more than ecosystem breadth.